saroj sapkota
A00025538

The Smart Travel Assistant (STA) is a small-scale, classical AI system built as a university coursework project. It uses a rule-based engine, a Bayesian probabilistic recommender, and a keyword-based intent classifier to help users choose a travel destination based on three preferences — budget, climate, and activity type. The system covers seven destinations and explains every recommendation it makes, showing the user exactly which preferences matched, which rules fired, and how every destination scored against each other.

In [ ]:
import re
# KNOWLEDGE BASE
# Static dictionary acting as the agent's database.
# All scoring, rules, and explanations pull from here.
knowledge_base = {
    "Paris": {
        "budget_level": "mid",
        "climate": "mixed",
        "best_seasons": ["spring", "autumn"],
        "safety_rating": 4,
        "transport": {"flight": 120, "train": 80},
        "activities": ["culture", "food", "sightseeing"],
        "description": "A romantic city known for art, cuisine, and iconic landmarks like the Eiffel Tower."
    },
    "Tokyo": {
        "budget_level": "luxury",
        "climate": "mixed",
        "best_seasons": ["spring", "autumn"],
        "safety_rating": 5,
        "transport": {"flight": 700},
        "activities": ["culture", "technology", "food"],
        "description": "A futuristic city blending ancient temples with cutting-edge technology and amazing food."
    },
    "Bangkok": {
        "budget_level": "budget",
        "climate": "warm",
        "best_seasons": ["winter"],
        "safety_rating": 2,
        "transport": {"flight": 500},
        "activities": ["markets", "food", "temples"],
        "description": "A vibrant city with bustling street markets, ornate temples, and cheap street food."
    },
    "New York": {
        "budget_level": "luxury",
        "climate": "mixed",
        "best_seasons": ["spring", "autumn"],
        "safety_rating": 3,
        "transport": {"flight": 450},
        "activities": ["shopping", "theatre", "sightseeing"],
        "description": "The city that never sleeps — famous for Broadway, skyscrapers, and world-class shopping."
    },
    "Dubai": {
        "budget_level": "luxury",
        "climate": "warm",
        "best_seasons": ["winter"],
        "safety_rating": 4,
        "transport": {"flight": 400},
        "activities": ["shopping", "desert", "luxury"],
        "description": "A modern desert city famous for luxury hotels, desert safaris, and massive shopping malls."
    },
    "Sydney": {
        "budget_level": "mid",
        "climate": "warm",
        "best_seasons": ["summer"],
        "safety_rating": 5,
        "transport": {"flight": 900},
        "activities": ["beach", "nature", "sightseeing"],
        "description": "A laid-back coastal city with stunning beaches, the Opera House, and great outdoor life."
    },
    # FIX (Option B): Added London to the knowledge base so it can be
    # honestly recommended, scored, and explained like any other destination.
    # Without this, the agent would either ignore the user's request or lie
    # by labelling a Sydney result as "London was recommended."
    "London": {
        "budget_level": "luxury",
        "climate": "mixed",
        "best_seasons": ["spring", "summer"],
        "safety_rating": 4,
        "transport": {"train": 50, "flight": 80},
        "activities": ["culture", "theatre", "sightseeing", "food"],
        "description": "A historic global city with world-class museums, theatre, diverse food, and iconic landmarks."
    }
}
 
 
# INTENT CLASSIFIER
 
class IntentClassifier:
    """
    Classifies user input using keyword scoring (bag-of-words).
    No ML needed — just checks which trigger words appear in the text.
    Fast, transparent, and easy to extend by adding more keywords.
    """
 
    def __init__(self):
        # Each intent has a list of trigger words.
        # The more words match, the higher that intent scores.
        self.intents = {
            "destination_query": ["where", "go", "travel", "holiday", "visit", "recommend", "suggest", "destination", "want to"],
            "cost_query":        ["cost", "price", "budget", "cheap", "expensive", "afford", "money"],
            "transport_query":   ["flight", "train", "transport", "fly", "get there", "travel to"],
            "season_query":      ["when", "season", "best time", "month", "time of year"],
            "safety_query":      ["safe", "danger", "dangerous", "crime", "risk", "secure"],
            "explain_query":     ["why", "explain", "reason", "how", "choose", "chose", "picked", "recommend", "tell me more", "what made"]
        }
 
    def classify(self, text):
        scores = {k: 0 for k in self.intents}
        text_lower = text.lower()
 
        for intent, words in self.intents.items():
            for w in words:
                if w in text_lower:
                    scores[intent] += 1
 
        best = max(scores, key=scores.get)
        total = sum(scores.values())
 
        # Confidence = fraction of total keyword hits that belong to the winning intent.
        # Low confidence means input is ambiguous — better to ask than guess wrong.
        confidence = scores[best] / total if total else 0
 
        if confidence < 0.35:
            return "unknown", confidence
 
        return best, confidence
 
 
# ACTIVITY MATCHER
def match_activity(user_input):
    """
    Fuzzy-matches the user's activity input to valid activity keywords in the knowledge base.
    This fixes the bug where typing 'roaming' scores 0 against every destination
    because no destination has 'roaming' — it silently penalises everyone equally.
 
    Instead we map common synonyms to the actual activity tags used in the knowledge base.
    """
    user_input = user_input.lower().strip()
 
    # Maps common words a user might type -> actual activity tag in the knowledge base
    activity_map = {
        "food": "food", "eating": "food", "restaurants": "food", "cuisine": "food", "eat": "food",
        "culture": "culture", "history": "culture", "museum": "culture", "art": "culture", "historical": "culture",
        "shopping": "shopping", "shop": "shopping", "malls": "shopping", "markets": "markets", "market": "markets",
        "beach": "beach", "beaches": "beach", "swimming": "beach", "sea": "beach", "ocean": "beach",
        "nature": "nature", "hiking": "nature", "outdoors": "nature", "wildlife": "nature", "roaming": "nature",
        "sightseeing": "sightseeing", "tourism": "sightseeing", "landmarks": "sightseeing", "tourist": "sightseeing",
        "technology": "technology", "tech": "technology", "gadgets": "technology",
        "temples": "temples", "religious": "temples", "spiritual": "temples",
        "theatre": "theatre", "theater": "theatre", "shows": "theatre", "broadway": "theatre",
        "desert": "desert", "safari": "desert", "dunes": "desert",
        "luxury": "luxury", "fancy": "luxury", "five star": "luxury", "5 star": "luxury"
    }
 
    # Direct match first
    if user_input in activity_map:
        return activity_map[user_input]
 
    # Partial match — check if any key is contained within what the user typed
    for key, tag in activity_map.items():
        if key in user_input:
            return tag
 
    # Return as-is if no match found — recommender will just penalise all equally
    return user_input
 
 
# CITY NAME DETECTOR
def detect_named_city(text, kb):
    """
    FIX (Option A): Checks if the user already named a specific destination
    in their input before we ask them preference questions.
 
    Previously the destination_query handler ignored city names in the input
    entirely and jumped straight to collecting budget/climate/activity.
    That meant "I want to visit London" triggered the recommender with no
    London-specific weighting — it was as if the user never said London at all.
 
    Now we:
      1. Check if any KB city appears in the user's message (case-insensitive).
      2. If it does and it's in the KB -> acknowledge it, set it as the
         preferred destination, and still run the full preference flow so
         the user can see how it compares to alternatives.
      3. If the city is NOT in the KB -> tell the user honestly and list
         what destinations we do have data for.
 
    Returns: (city_name_or_None, in_kb_bool)
    """
    text_lower = text.lower()
    for city in kb:
        if city.lower() in text_lower:
            return city, True
 
    # Also check some common cities not in KB so we can give a helpful message
    unknown_cities = [
        "nepal", "kathmandu", "mumbai", "delhi", "berlin", "rome", "amsterdam",
        "singapore", "beijing", "shanghai", "los angeles", "chicago", "toronto",
        "madrid", "barcelona", "lisbon", "cairo", "nairobi", "cape town"
    ]
    for city in unknown_cities:
        if city in text_lower:
            # Return the city name with title case and flag it as not in KB
            return city.title(), False
 
    return None, False
 
 
# BAYESIAN RECOMMENDER
class Recommender:
    """
    Scores destinations using a Bayesian-style multiplier system.
 
    Each destination starts at 1.0 (equal probability).
    Matching a preference multiplies score by 1.5 (boost).
    Mismatching multiplies by 0.7 or 0.8 (penalty).
 
    Multiplication compounds — 3 mismatches = 0.7 * 0.7 * 0.7 = 0.34,
    which correctly pushes poor-fit destinations far down the list.
 
    If a preferred_city is passed, that city gets an additional 2.0x boost
    to reflect the user's explicit stated preference. This means the city
    they named will rank first unless their preferences strongly contradict
    it (e.g. asking for budget travel but naming Tokyo, which is luxury).
    """
 
    def __init__(self, kb):
        self.kb = kb
 
    def recommend(self, prefs, preferred_city=None):
        scores = {d: 1.0 for d in self.kb}
 
        for d, data in self.kb.items():
            if "budget" in prefs:
                scores[d] *= 1.5 if prefs["budget"] == data["budget_level"] else 0.7
 
            if "climate" in prefs:
                scores[d] *= 1.5 if prefs["climate"] == data["climate"] else 0.7
 
            if "activity" in prefs:
                # Softer penalty for activity (0.8 not 0.7) — activity is more flexible than budget
                scores[d] *= 1.5 if prefs["activity"] in data["activities"] else 0.8
 
        # Apply explicit city preference boost AFTER preference scoring.
        # Why 2.0x? It's strong enough to respect the user's stated wish
        # but not so dominant that preferences are ignored entirely.
        # A city that contradicts all 3 preferences still gets pushed down.
        if preferred_city and preferred_city in scores:
            scores[preferred_city] *= 2.0
 
        # Normalise so all scores sum to 1.0 — makes them readable as percentages
        total = sum(scores.values())
        results = [(d, round(scores[d] / total, 2)) for d in scores]
 
        return sorted(results, key=lambda x: x[1], reverse=True)
 
 
# RULE ENGINE
class RuleEngine:
    """
    Hard-coded IF-THEN rules that encode travel expertise.
    These fire on top of the Bayesian scorer and are shown in explanations
    so the user understands domain logic, not just probability numbers.
    """
 
    def apply(self, prefs):
        rules = []
 
        # Rule 1: Budget + warm is the classic Bangkok combo
        if prefs.get("budget") == "budget" and prefs.get("climate") == "warm":
            rules.append("Your budget + warm climate preference is a strong match for Bangkok.")
 
        # Rule 2: Warm climate in general points toward Dubai or Sydney
        if prefs.get("climate") == "warm":
            rules.append("Warm climate preference also makes Dubai and Sydney strong candidates.")
 
        # Rule 3: Luxury budget with mixed climate fits Tokyo or New York
        if prefs.get("budget") == "luxury" and prefs.get("climate") == "mixed":
            rules.append("Luxury budget + mixed climate is a strong fit for Tokyo and New York.")
 
        # Rule 4: Mid budget with nature/beach points to Sydney
        if prefs.get("budget") == "mid" and prefs.get("activity") in ["beach", "nature"]:
            rules.append("Mid budget + nature/beach activity makes Sydney an excellent pick.")
 
        # Rule 5: Luxury + mixed + culture/theatre/sightseeing is a London fit
        if prefs.get("budget") == "luxury" and prefs.get("activity") in ["culture", "theatre", "sightseeing", "food"]:
            rules.append("Luxury budget + cultural/theatre activity is well suited to London.")
 
        return rules
 
 
# EXPLAINER
class Explainer:
    """
    Makes the recommendation transparent.
    Shows what preferences were used, which rules fired, why the score is what it is,
    and what the destination actually offers — so the user can challenge or accept it.
    """
 
    def explain(self, dest, prefs, score, rules, kb, all_results, preferred_city=None):
        dest_data = kb[dest]
 
        print(f"\n{'='*45}")
        print(f"  Why {dest} was recommended")
        print(f"{'='*45}")
 
        # If user named a city and it WAS recommended — explain the boost
        if preferred_city and preferred_city == dest:
            print(f"\n  Note: You specifically requested {dest}, so it received a preference boost.")
            print("  Your stated preferences were still scored — they determine how well it fits.")
 
        # If user named a city but a DIFFERENT city was recommended — explain why
        if preferred_city and preferred_city != dest:
            print(f"\n  Note: You mentioned {preferred_city}, but your preferences pointed more strongly to {dest}.")
            print(f"  {preferred_city} still received a preference boost — but the mismatch with your budget/climate/activity outweighed it.")
 
        # Section 1: What you told the agent
        print("\n  Your preferences:")
        print(f"    - Budget:   {prefs.get('budget', 'not specified')}")
        print(f"    - Climate:  {prefs.get('climate', 'not specified')}")
        print(f"    - Activity: {prefs.get('activity', 'not specified')}")
 
        # Section 2: How well the destination matched each preference
        print(f"\n  How {dest} matched your preferences:")
 
        budget_match = prefs.get("budget") == dest_data["budget_level"]
        climate_match = prefs.get("climate") == dest_data["climate"]
        activity_match = prefs.get("activity") in dest_data["activities"]
 
        print(f"    - Budget ({dest_data['budget_level']}):   {'✓ match  (+50% score)' if budget_match else '✗ no match (-30% score)'}")
        print(f"    - Climate ({dest_data['climate']}):  {'✓ match  (+50% score)' if climate_match else '✗ no match (-30% score)'}")
        print(f"    - Activity ({', '.join(dest_data['activities'])}): {'✓ match  (+50% score)' if activity_match else '✗ no match (-20% score)'}")
 
        # Section 3: Rules that fired
        if rules:
            print("\n  Rules triggered:")
            for r in rules:
                print(f"    -> {r}")
        else:
            print("\n  No specific rules triggered for this combination.")
 
        # Section 4: Final score and safety
        print(f"\n  Final match score: {score * 100:.0f}%")
        print(f"  Safety rating: {dest_data['safety_rating']}/5")
        if dest_data["safety_rating"] <= 2:
            print("  Warning: Low safety rating — research travel advisories before booking.")
 
        # Section 5: What the destination is actually like
        print(f"\n  About {dest}:")
        print(f"    {dest_data['description']}")
 
        # Section 6: Transport options
        print(f"\n  Transport from UK:")
        for mode, cost in dest_data["transport"].items():
            print(f"    - {mode.capitalize()}: ~£{cost}")
 
        # Section 7: Show how other destinations ranked
        print("\n  Full ranking:")
        for i, (d, s) in enumerate(all_results):
            bar = "█" * int(s * 30)
            marker = " <- recommended" if d == dest else ""
            pref_marker = " (your choice)" if d == preferred_city and preferred_city != dest else ""
            print(f"    {i+1}. {d:<12} {bar:<30} {s*100:.0f}%{marker}{pref_marker}")
 
        print(f"\n{'='*45}\n")
 
 
# TRAVEL INFO HANDLER
class TravelInfo:
    """
    Handles direct travel questions that aren't recommendations.
    If the user asks about cost, transport, safety, or best season
    for the last recommended destination, this provides the answer.
    """
 
    def answer(self, intent, memory, kb):
        if not memory:
            print("Please ask for a destination recommendation first, then I can answer questions about it.")
            return
 
        dest = memory["last_dest"]
        data = kb[dest]
 
        if intent == "cost_query":
            print(f"\n  Cost info for {dest}:")
            for mode, cost in data["transport"].items():
                print(f"    - {mode.capitalize()} from UK: ~£{cost}")
            print(f"    - Budget level: {data['budget_level']} (relative to other destinations)")
 
        elif intent == "transport_query":
            print(f"\n  Transport options to {dest}:")
            for mode, cost in data["transport"].items():
                print(f"    - {mode.capitalize()}: ~£{cost}")
            if len(data["transport"]) == 1:
                print("    - Only one transport option available from the UK.")
 
        elif intent == "safety_query":
            rating = data["safety_rating"]
            print(f"\n  Safety info for {dest}:")
            print(f"    - Safety rating: {rating}/5")
            if rating >= 4:
                print("    - Generally considered safe for tourists.")
            elif rating == 3:
                print("    - Moderate safety — stay alert in busy areas.")
            else:
                print("    - Lower safety rating — check government travel advisories before booking.")
 
        elif intent == "season_query":
            seasons = data["best_seasons"]
            print(f"\n  Best time to visit {dest}:")
            print(f"    - Recommended seasons: {', '.join(seasons)}")
 
 
# MAIN AGENT
class TravelAgent:
    """
    Top-level controller that coordinates all components.
    Each component handles one job — this class just decides which one to call.
    """
 
    def __init__(self):
        self.kb = knowledge_base
        self.classifier = IntentClassifier()
        self.recommender = Recommender(self.kb)
        self.rules = RuleEngine()
        self.explainer = Explainer()
        self.travel_info = TravelInfo()
        self.memory = {}
        self.all_results = []
 
    def run(self):
        print("\nWelcome to Smart Travel Assistant!")
        print("--------------------------------------------------")
        print("You can ask things like:")
        print(" - 'Where should I go?'")
        print(" - 'I want to visit London'")
        print(" - 'Why choose Paris?'")
        print(" - 'Cost of Tokyo trip'")
        print(" - 'Is it safe?'")
        print(" - 'Best season for travel'")
        print("--------------------------------------------------")
        print(f"Known destinations: {', '.join(self.kb.keys())}")
        print("Tip: Type clearly for better results.")
        print("Type 'quit' anytime to exit.\n")
 
        while True:
            user = input("You: ").strip()
 
            if user.lower() in ["quit", "bye", "exit"]:
                print("Goodbye! Safe travels.")
                break
 
            if not user:
                print("Please type something.")
                continue
 
            # EXPLAIN CHECK FIRST — before the classifier runs.
            # Intercept explain-words before classifier so they don't fall
            # through to "unknown" when confidence is low.
            explain_words = [
                "why", "explain", "reason", "how did you", "why did you",
                "why choose", "why recommend", "tell me more", "what made",
                "why this", "why that", "justify", "elaborate"
            ]
 
            if any(word in user.lower() for word in explain_words):
                if self.memory:
                    self.explainer.explain(
                        self.memory["last_dest"],
                        self.memory["prefs"],
                        self.memory["score"],
                        self.memory["rules"],
                        self.kb,
                        self.all_results,
                        preferred_city=self.memory.get("preferred_city")
                    )
                else:
                    print("No recommendation yet. Ask me where to travel first.")
                continue
 
            # CLASSIFIER
            intent, conf = self.classifier.classify(user)
 
            # DESTINATION QUERY — full recommendation flow
            if intent == "destination_query":
                prefs = {}
                preferred_city = None
 
                # FIX: Detect if the user named a specific city BEFORE collecting preferences.
                # Previously this was never checked — the agent jumped straight to budget/climate/activity
                # and the named city was completely ignored.
                named_city, in_kb = detect_named_city(user, self.kb)
 
                if named_city and in_kb:
                    # City is known — acknowledge it, apply a boost in recommender
                    print(f"\nYou mentioned {named_city} — I have data for that destination.")
                    print("I'll still ask your preferences so we can compare it fairly against alternatives.")
                    preferred_city = named_city
 
                elif named_city and not in_kb:
                    # City is unknown — be honest, don't pretend to know it
                    print(f"\nSorry, I don't have data for {named_city}.")
                    print(f"I can compare: {', '.join(self.kb.keys())}")
                    print("I'll find the best match from these based on your preferences.\n")
 
                print("\nLet me find the best destination for you.")
 
                # Validate budget input
                while True:
                    budget = input("Budget (budget / mid / luxury): ").strip().lower()
                    if budget in ["budget", "mid", "luxury"]:
                        prefs["budget"] = budget
                        break
                    print("  Please enter: budget, mid, or luxury")
 
                # Validate climate input
                while True:
                    climate = input("Climate preference (warm / cold / mixed): ").strip().lower()
                    if climate in ["warm", "cold", "mixed"]:
                        prefs["climate"] = climate
                        break
                    print("  Please enter: warm, cold, or mixed")
 
                # Activity — fuzzy matched
                raw_activity = input("Activity (e.g. food, beach, culture, shopping, nature, temples): ").strip()
                prefs["activity"] = match_activity(raw_activity)
 
                if prefs["activity"] != raw_activity.lower():
                    print(f"  (Matched '{raw_activity}' -> '{prefs['activity']}')")
 
                # Run recommender — pass preferred_city so it gets the 2x boost
                self.all_results = self.recommender.recommend(prefs, preferred_city=preferred_city)
                top = self.all_results[0]
 
                print(f"\n  Top Recommendation: {top[0]}  ({top[1] * 100:.0f}% match)")
                print(f"  {self.kb[top[0]]['description']}")
 
                # Safety warning
                if self.kb[top[0]]["safety_rating"] <= 2:
                    print("  Warning: This destination has a lower safety rating.")
 
                # Close call — two destinations are very close in score
                if abs(self.all_results[0][1] - self.all_results[1][1]) < 0.05:
                    print(f"  Close call: {self.all_results[1][0]} is also a strong match.")
 
                # If user named a city but got a different recommendation — surface it
                if preferred_city and top[0] != preferred_city:
                    # Find where the preferred city ranked
                    pref_rank = next((i+1 for i, (d, _) in enumerate(self.all_results) if d == preferred_city), None)
                    pref_score = next((s for d, s in self.all_results if d == preferred_city), None)
                    print(f"\n  Note: {preferred_city} ranked #{pref_rank} ({pref_score*100:.0f}% match).")
                    print(f"  Your preferences were a stronger fit for {top[0]}.")
                    print(f"  Type 'explain' to see why.")
                else:
                    print("\n  Type 'why' or 'explain' to see the full reasoning.")
 
                # Save to memory for follow-up questions
                self.memory = {
                    "last_dest":      top[0],
                    "prefs":          prefs,
                    "score":          top[1],
                    "rules":          self.rules.apply(prefs),
                    "preferred_city": preferred_city
                }
 
            # FOLLOW-UP QUESTIONS
            elif intent in ["cost_query", "transport_query", "safety_query", "season_query"]:
                self.travel_info.answer(intent, self.memory, self.kb)
 
            # UNKNOWN
            elif intent == "unknown":
                print("I didn't quite understand that.")
                print("Try: 'where should I travel?', 'is it safe?', 'how much does it cost?', or 'why did you choose that?'")
 
            # FALLBACK
            else:
                print("I can help with: destination recommendations, cost, transport, safety, best season, and explanations.")
 
 
# RUN
if __name__ == "__main__":
    agent = TravelAgent()
    agent.run()


Welcome to Smart Travel Assistant!
--------------------------------------------------
You can ask things like:
 - 'Where should I go?'
 - 'I want to visit London'
 - 'Why choose Paris?'
 - 'Cost of Tokyo trip'
 - 'Is it safe?'
 - 'Best season for travel'
--------------------------------------------------
Known destinations: Paris, Tokyo, Bangkok, New York, Dubai, Sydney, London
Tip: Type clearly for better results.
Type 'quit' anytime to exit.



You:  i want to visit london



You mentioned London — I have data for that destination.
I'll still ask your preferences so we can compare it fairly against alternatives.

Let me find the best destination for you.


Budget (budget / mid / luxury):  mid
Climate preference (warm / cold / mixed):  warm
Activity (e.g. food, beach, culture, shopping, nature, temples):  nature



  Top Recommendation: Sydney  (45% match)
  A laid-back coastal city with stunning beaches, the Opera House, and great outdoor life.

  Note: London ranked #5 (11% match).
  Your preferences were a stronger fit for Sydney.
  Type 'explain' to see why.
